# Loading Data

In [ ]:
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from sklearn.preprocessing import MinMaxScaler
import torch
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import precision_recall_curve
import numpy as np
import torch.nn as nn
from sklearn.metrics import classification_report, roc_auc_score
import joblib
from sklearn.metrics import confusion_matrix

In [ ]:
normal = pd.read_csv("../normal.csv")
attack = pd.read_csv("../attack.csv")
merged = pd.concat([normal, attack], ignore_index=True)

# Data Exploring

In [ ]:
merged.head()

In [ ]:
merged.info()

In [ ]:
merged.value_counts().sum()

In [ ]:
merged.isnull().sum()

In [ ]:
merged["attack"].value_counts()

**total=1441719 then all labeled**

In [ ]:
merged.duplicated().sum()

1. Missing Values Pattern is Intentional
In SWaT, missing values often indicate:

Pumps/Valves being OFF (binary states like MV101, P201)

Sensors not recording during certain operational phases

Normal operational patterns (e.g., some components only active in specific treatment stages)

2. Duplicates are Expected
SWaT data is collected at regular intervals, but:

Same readings can occur during stable operation

Attack scenarios might create repeated patterns

Removing duplicates is safe as they don't add information

# Data Cleaning

In [ ]:
def complete_swat_preprocessing(merged):

    print(f"Initial shape: {merged.shape}")
    print(f"Columns: {merged.columns.tolist()}")
    
    df_proc = merged.copy()
    
    # Fix column names - strip whitespace
    df_proc.columns = df_proc.columns.str.strip()
    
    # Step 2: Remove duplicates (safe for SWaT)
    initial_len = len(df_proc)
    df_proc = df_proc.drop_duplicates()
    print(f"Removed {initial_len - len(df_proc)} duplicate rows")
    
    # Step 3: Separate features and target
    target = df_proc['attack']
    features = df_proc.drop(columns=['attack','Timestamp'])
    
    # Step 4: Fix column names in features (strip whitespace)
    features.columns = features.columns.str.strip()
    
    # Step 5: Handle missing values based on SWaT domain knowledge
    # Binary actuators (fill with 0 for OFF state)
    binary_actuators = ['MV101', 'MV201', 'P201', 'P202', 'P204', 'MV303']
    for col in binary_actuators:
        matching_cols = [c for c in features.columns if c.strip() == col]
        if matching_cols:
            actual_col = matching_cols[0]
            features[actual_col] = features[actual_col].fillna(0)
            features[actual_col] = features[actual_col].astype(int)
            if actual_col != col:
                features = features.rename(columns={actual_col: col})
    
    # Critical continuous sensors (interpolate)
    critical_sensors = ['LIT101', 'AIT201', 'AIT202', 'FIT401', 'PIT501']
    for col in critical_sensors:
        matching_cols = [c for c in features.columns if c.strip() == col]
        if matching_cols:
            actual_col = matching_cols[0]
            features[actual_col] = features[actual_col].interpolate(method='linear', limit_direction='both') #Linear interpolation fills missing values based on surrounding values.
            if actual_col != col:
                features = features.rename(columns={actual_col: col})
    
    # Other sensors (fill with forward fill then median)
    numeric_cols = features.select_dtypes(include=[np.number]).columns
    other_sensors = [col for col in numeric_cols if col not in binary_actuators + critical_sensors]
    for col in other_sensors:
        features[col] = features[col].ffill().fillna(features[col].median())
    
    
    # Step 7: Encode target
    if target.dtype == 'object':
        target_encoded = (target.str.strip() == 'Attack').astype(int)
    else:
        target_encoded = target
    
    print(f"\nFinal shape: {features.shape}")
    print(f"Missing values: {features.isnull().sum().sum()}")
    print(f"Attack samples: {target_encoded.sum()}")
    print(f"Normal samples: {len(target_encoded) - target_encoded.sum()}")
    
    return features, target_encoded

In [ ]:
X_swat, y_swat = complete_swat_preprocessing(merged)


print(f" Features shape: {X_swat.shape}")
print(f" Target shape: {y_swat.shape}")
print(f" Attack samples: {y_swat.sum()} ({y_swat.mean()*100:.2f}%)")
print(f" Normal samples: {len(y_swat) - y_swat.sum()} ({(1-y_swat.mean())*100:.2f}%)")
print(f" Missing values in features: {X_swat.isnull().sum().sum()}")

**Preprocessing Results Summary:**

Original rows:	1,441,719

After removing duplicates:	946,719

Duplicates removed:	495,000 (34.3%)

Features (columns):	57

Missing values:	0 

Attack samples:	54,621 (5.77%)

Normal samples:	892,098 (94.23%)

Time range:	Dec 22, 2015 - Jan 2, 2016

**For USAD, typically:**

**Train mostly on Normal data**

**Test on Normal + Attack**

If shuffled:

The model may see future patterns

Data leakage happens

In [ ]:
#finding attack indecies
attack_indices = y_swat[y_swat == 1].index

print("First attack:", attack_indices.min())
print("Last attack:", attack_indices.max())
print("Total attacks:", len(attack_indices))

# Splitting the data

In [ ]:
split_index = int(len(X_swat) * 0.7)

X_train = X_swat.iloc[:split_index]
y_train = y_swat.iloc[:split_index]

X_test = X_swat.iloc[split_index:]
y_test = y_swat.iloc[split_index:]

In [ ]:
print("Train attacks:", y_train.sum())
print("Test attacks:", y_test.sum())

this is perfect for USAD beacause now i have Train mostly on Normal data and

Test on Normal + Attack

# Normalization

In [ ]:
scaler = MinMaxScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Preparing for the USAD

In [ ]:
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)

In [ ]:
train_dataset = TensorDataset(X_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=1024, shuffle=False)

In [ ]:
next(iter(train_loader))[0].shape

In [ ]:
input_dim = X_train.shape[1]  # should be 51
print(input_dim)

**so the neural should be:**

Input Layer: 51 neurons

↓

Encoder

↓

Latent Space

↓

Decoder1

Decoder2


# Implementing the model

In [ ]:
input_dim = 51
hidden_dim = 25
latent_dim = 12


class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, latent_dim),
            nn.ReLU()
        )

    def forward(self, x):
        return self.net(x)


class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim),
            nn.Sigmoid()
        )

    def forward(self, z):
        return self.net(z)
    

In [ ]:
def set_seed(seed_value=42):
    """
    Sets random seeds for reproducibility across Python, NumPy, and PyTorch.
    """
    os.environ["PYTHONHASHSEED"] = str(seed_value)
    random.seed(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    torch.cuda.manual_seed(seed_value) # For all GPUs
    torch.cuda.manual_seed_all(seed_value) # For multiple GPUs
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"Random seed set to {seed_value}")
    return set_seed

In [ ]:
class USAD(nn.Module):

    def __init__(self):
        super().__init__()

        self.encoder = Encoder()
        self.decoder1 = Decoder()
        self.decoder2 = Decoder()

    def forward(self, x):

        z = self.encoder(x)

        w1 = self.decoder1(z)
        w2 = self.decoder2(z)

        z2 = self.encoder(w1)
        w3 = self.decoder2(z2)

        return w1, w2, w3

In [ ]:
import random
import os
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = USAD().to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)

epochs = 30
set_seed(42)

In [ ]:
for epoch in range(epochs):

    model.train()
    total_loss = 0

    for batch in train_loader:

        x = batch[0].to(device)

        w1, w2, w3 = model(x)

        mse1 = torch.mean((x - w1)**2)
        mse2 = torch.mean((x - w2)**2)
        mse3 = torch.mean((x - w3)**2)

        n = epoch + 1

        loss1 = (1/n)*mse1 + (1 - 1/n)*mse3
        loss2 = (1/n)*mse2 - (1 - 1/n)*mse3

        loss = loss1 + loss2

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs} Loss: {total_loss:.4f}")

In [ ]:
model.eval()

X_test_tensor = X_test_tensor.to(device)

with torch.no_grad():

    w1, w2, w3 = model(X_test_tensor)

    error1 = torch.mean((X_test_tensor - w1) ** 2, dim=1)
    error2 = torch.mean((X_test_tensor - w3) ** 2, dim=1)

    anomaly_score = 0.7*error1 + 0.3*error2 #weighted score

anomaly_score = anomaly_score.cpu().numpy()

# Find the best threshold automatically

In [ ]:
precision, recall, thresholds = precision_recall_curve(y_test, anomaly_score)

f1 = 2 * (precision * recall) / (precision + recall + 1e-8)
best_idx = np.argmax(f1)

best_threshold = thresholds[best_idx]

print("Best threshold:", best_threshold)
print("Best F1:", f1[best_idx])

In [ ]:
y_pred = (anomaly_score > best_threshold).astype(int)

# Evaluation

In [ ]:
print(classification_report(y_test, y_pred))

roc = roc_auc_score(y_test, anomaly_score)

print("ROC AUC:", roc)

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
cm= confusion_matrix(y_test, y_pred)
print(confusion_matrix(y_test, y_pred))

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=['Normal', 'Attack'], yticklabels=['Normal', 'Attack '])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
plt.figure(figsize=(12,5))

plt.plot(anomaly_score, label="Anomaly Score")
plt.axhline(best_threshold, color="red", linestyle="--", label="Threshold")

plt.legend()
plt.title("USAD Anomaly Detection")

plt.show()

In [ ]:
from sklearn.metrics import roc_curve
fpr, tpr, thresholds = roc_curve(y_test, anomaly_score)

optimal_idx = np.argmax(tpr - fpr)
optimal_threshold = thresholds[optimal_idx]

In [ ]:
plt.figure(figsize=(8,6))
plt.plot(fpr, tpr, label=f"ROC Curve (AUC = {roc:.3f})")
plt.plot([0,1],[0,1],'k--',label="Random classifier")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - USAD on SWaT")
plt.legend()
plt.grid(True)

plt.show()

# Saving

In [ ]:
torch.save(model.state_dict(), "usad_swat_weights.pth")

In [ ]:
torch.save(model, "usad_swat_model_full.pth")

In [ ]:
joblib.dump(scaler, "swat_scaler.pkl")

In [ ]:
# --- Test ONLY on attack.csv ---
print("Evaluating USAD ONLY on attack.csv...")
import pandas as pd
import numpy as np
import torch
from sklearn.metrics import classification_report, confusion_matrix

atk_df = pd.read_csv("../attack.csv")
label_col = 'attack'
feature_cols = [c for c in atk_df.columns if c != label_col and c != 'Timestamp' and c != 'label']
X_atk = atk_df[feature_cols].select_dtypes(include=[np.number]).values
X_atk = np.nan_to_num(X_atk, nan=0.0)

# The scaler is already trained as 'scaler'
X_atk_scaled = scaler.transform(X_atk)

y_raw_atk = atk_df[label_col].values
if y_raw_atk.dtype == object or isinstance(y_raw_atk[0], str):
    y_atk = np.where(y_raw_atk == 'Normal', 0, 1).astype(int)
else:
    y_atk = (y_raw_atk != 0).astype(int)

X_atk_tensor = torch.tensor(X_atk_scaled, dtype=torch.float32).to(device)

model.eval()
with torch.no_grad():
    w1_atk, w2_atk, w3_atk = model(X_atk_tensor)
    error1_atk = torch.mean((X_atk_tensor - w1_atk) ** 2, dim=1)
    error2_atk = torch.mean((X_atk_tensor - w3_atk) ** 2, dim=1)
    anomaly_score_atk = 0.7 * error1_atk + 0.3 * error2_atk

anomaly_score_atk = anomaly_score_atk.cpu().numpy()

# threshold is 'best_threshold'
y_pred_atk = (anomaly_score_atk > best_threshold).astype(int)

print("Classification Report on attack.csv:")
print(classification_report(y_atk, y_pred_atk))
print("Confusion Matrix on attack.csv:")
print(confusion_matrix(y_atk, y_pred_atk))
